# Контрольное мероприятие 1

### 1) загрузите БД Innovations.xlsx и подготовьте ее к работе (БД вы найдет в папке на GD, там же будет доп. документ с описанием исходных переменных)

1.1 Добавить блок с цитатой

1.2 - переведите названия колонок на русский язык

1.3 - проверьте уровень измерения переменных, если есть числовые переменные не обозначенные так автоматически, выполните настройку вручную

1.4 - проверьте наличие пропусков, сделайте вывод о пригодности данных для дальнейшего анализа

In [15]:
import pandas as pd

df = pd.read_excel("../KT_for_PSH/Innovations.xlsx", sheet_name='Лист1')

In [2]:
df.head()


,Субъект РФ,Potential,Patients,Technologies,Organizations,Employes,Researchers,Students,Research costs,Organizational expenses,Products
0,Белгородская область,0.4088,261,2408,21.507761,1655,489,316,1921.0,23852.3910,11.6
1,Брянская область,0.3197,147,1603,14.326648,688,78,223,977.7,1466.8642,7.3
2,Владимирская область,0.3530,292,6728,24.873096,5365,308,197,5391.3,6077.5891,8.1
3,Воронежская область,0.4089,594,2538,21.301775,10654,947,373,8164.5,13518.6942,6.1
4,Ивановская область,0.3226,448,933,13.705584,574,233,260,585.7,253.3257,0.2


In [3]:
df.columns = [
    "Субъект РФ", "Потенциал", "Пациенты", "Технологии", "Организации",
    "Сотрудники", "Исследователи", "Студенты", "Расходы на исследования",
    "Организационные расходы", "Продукция"
]

In [4]:
df.dtypes

Субъект РФ                  object
Потенциал                  float64
Пациенты                     int64
Технологии                   int64
Организации                float64
Сотрудники                   int64
Исследователи                int64
Студенты                     int64
Расходы на исследования    float64
Организационные расходы    float64
Продукция                  float64
dtype: object

In [5]:
columns_numeric = df.columns.drop("Субъект РФ")  # кроме названия региона
df[columns_numeric] = df[columns_numeric].apply(pd.to_numeric, errors="coerce") # Принудительно преобразуем числовые переменные

In [6]:
df.isna().sum()


Субъект РФ                 0
Потенциал                  0
Пациенты                   0
Технологии                 0
Организации                0
Сотрудники                 0
Исследователи              0
Студенты                   0
Расходы на исследования    0
Организационные расходы    0
Продукция                  0
dtype: int64

**Пустых значений нет, данные можно использовать**

### 2) На основе таблицы с данными для всех показателей (столбцов), измеренных в шкале интервалов (количественная), а также с типом данных float64 и int64 рассчитайте следующие описательные статистики.

Оформить в виде единой таблицы:

**Среднее Стандартная ошибка среднего Медиана Мода Стандартное отклонение Минимум Максимум Асимметрия Эксцесс**

Результат сохраните в виде таблицы формата *.csv или *.xlsx.

2.1. Проанализировать данные любого столбца, измеренного в шкале интервалов, на наличие пропусков. Определить, есть ли они и если есть, то сколько.

2.2. Проанализировать данные на наличие выбросов. Используйте следующие способы устранения выбросов:

2.3. удаление конкретных значений (замена на Nan) удаление случаев (людей) из датафрейма замена на 3 или -3. 

2.4. Преобразовать столбец (Serie) в z-scores. 

2.5. Фильтровать на основе условия. Удалить или заменить выбросы на другие значения.

In [7]:

# Фильтрация только количественных переменных
numeric_df = df.select_dtypes(include=["float64", "int64"])

desc_stats = pd.DataFrame({
    'Среднее': numeric_df.mean(),
    'Станд. ошибка ср.': numeric_df.sem(),
    'Медиана': numeric_df.median(),
    'Мода': numeric_df.mode().iloc[0],
    'Ст. отклонение': numeric_df.std(),
    'Минимум': numeric_df.min(),
    'Максимум': numeric_df.max(),
    'Асимметрия': numeric_df.skew(),
    'Эксцесс': numeric_df.kurt()
})

In [9]:
desc_stats.to_excel("Описательные_статистики.xlsx") # Сохранение в .xlsx

In [10]:
df["Исследователи"].isnull().sum()

np.int64(0)

Пропусков нет

In [11]:
import numpy as np
from scipy import stats

# Преобразование в Z-оценки
z_scores = stats.zscore(df["Продукция"], nan_policy="omit")

# Создание новой колонки
df["Продукция_z"] = z_scores

# замена на NaN
df["Продукция_clean1"] = df["Продукция"].where(np.abs(z_scores) <= 3, np.nan)

# Удаление строк с выбросами
df_cleaned = df[np.abs(z_scores) <= 3]

# Замена выбросов на границу ±3
df["Продукция_clean3"] = np.where(
    z_scores > 3, df["Продукция"].mean() + 3*df["Продукция"].std(),
    np.where(z_scores < -3, df["Продукция"].mean() - 3*df["Продукция"].std(), df["Продукция"])
)

In [12]:
df["Продукция_z"] = stats.zscore(df["Продукция"], nan_policy="omit")

In [13]:
# Оставляем только значения с z в пределах [-3, 3]
filtered_df = df[np.abs(df["Продукция_z"]) <= 3]

In [14]:
filtered_df

,Субъект РФ,Потенциал,Пациенты,Технологии,Организации,Сотрудники,Исследователи,Студенты,Расходы на исследования,Организационные расходы,Продукция,Продукция_z,Продукция_clean1,Продукция_clean3
0,Белгородская область,0.4088,261,2408,21.507761,1655,489,316,1921.0,23852.3910,11.6,0.770182,11.6,11.6
1,Брянская область,0.3197,147,1603,14.326648,688,78,223,977.7,1466.8642,7.3,0.113777,7.3,7.3
2,Владимирская область,0.3530,292,6728,24.873096,5365,308,197,5391.3,6077.5891,8.1,0.235899,8.1,8.1
3,Воронежская область,0.4089,594,2538,21.301775,10654,947,373,8164.5,13518.6942,6.1,-0.069406,6.1,6.1
4,Ивановская область,0.3226,448,933,13.705584,574,233,260,585.7,253.3257,0.2,-0.970055,0.2,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Камчатский край,0.3002,13,324,34.210526,921,208,160,1205.6,437.0665,1.8,-0.725811,1.8,1.8
71,Приморский край,0.3373,260,1271,22.710623,5700,1645,255,6930.7,2087.7790,0.5,-0.924259,0.5,0.5
72,Хабаровский край,0.4077,183,2602,21.126761,1717,693,359,5898.0,8958.6558,23.8,2.632541,23.8,23.8
73,Амурская область,0.2423,115,696,14.285714,536,158,199,478.7,3817.0145,0.9,-0.863198,0.9,0.9
